In [22]:
import pandas as pd
import numpy as np
import pickle

In [84]:
from ortools.linear_solver import pywraplp


def get_data(
        df: pd.DataFrame,
) -> tuple[int, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n = df.shape[0]
    clicks = df['clicks'].values
    impressions = df['impressions'].values
    return n, clicks, impressions

def solver(
        df: pd.DataFrame,
        n: int,
        clicks: np.ndarray,
        impressions: np.ndarray,
        soglia_ctr: float = None,
) -> pd.DataFrame:
    solver = pywraplp.Solver.CreateSolver('SCIP')
    # Boolean variables
    x = [solver.BoolVar(f'x{i}') for i in range(n)]
    x_np = np.array(x)
    # Objective function
    solver.Maximize(np.dot(clicks, x_np))
    if soglia_ctr is not None:
        # CTR constraint
        solver.Add(np.dot(clicks, x_np) >= soglia_ctr * np.dot(impressions, x_np))
    # Solve the knapsack problem
    status = solver.Solve()
    results = pd.DataFrame(columns=df.columns)
    # Cycle over the boolean variables to get the selected rows
    if status == pywraplp.Solver.OPTIMAL:
        for i in range(n):
            if x[i].solution_value() == 1:
                if results.empty:
                    results = df.iloc[[i]]
                else:
                    results = pd.concat([results, df.iloc[[i]]])
        print("Knapsack Solver: Optimal solution!")
        print(f"Total clicks = {results['clicks'].sum()}")
        print(f"Total impressions = {results['impressions'].sum()}")
        print(f"Number of selected publishers = {results.shape[0]}")
        if soglia_ctr is not None:
            if results['impressions'].sum() != 0:
                print(f"CTR = {results['clicks'].sum() / results['impressions'].sum()}")
            else:
                print("CTR = undefined (division by zero)")
    else:
        print("Knapsack Solver: No feasible solution found.")
        # Return empty dataframe
        return pd.DataFrame()
    return results

def knapsack(
        df: pd.DataFrame,
        soglia_ctr: float = None,
) -> pd.DataFrame:
    n, clicks, impressions = get_data(df)
    return solver(df, n, clicks, impressions, soglia_ctr)

In [2]:
# Set up Random Number Generator
seed = 0
rng = np.random.default_rng(seed)
np.random.seed(seed)

In [3]:
# Parameters
num_iter = 500
rounds_per_iter = 200
embedding_size = 70

## Carico adv e publisher embedding

In [7]:
adv_embedding_path = '../src/publisher_embedding/data/embeddings_to_pick/ad_embeddings_red_70.pkl'
adv_embeddings = pickle.load(open(adv_embedding_path, 'rb'))

In [93]:
agents = [
    {
      "name": "Nostro",
      "adv_name": "Racer 1000",
    },
    {
      "name": "Reflex Pro",
      "adv_name": "Reflex Pro",
    },
    {
      "name": "Speedster GT",
      "adv_name": "Speedster GT",
    },
    {
      "name": "Pasta Bio",
      "adv_name": "Pasta Bio",
    },
    {
      "name": "Sneakers High-top",
      "adv_name": "Sneakers High-top"
    },
    {
      "name": "Conto Risparmio Plus",
      "adv_name": "Conto Risparmio Plus",
    },
    {
      "name": "Smartphone Pro X",
      "adv_name": "Smartphone Pro X",
    }
  ]

In [94]:
selected_adv_embeddings = {agent['adv_name']: adv_embeddings[agent['adv_name']] for agent in agents}

In [70]:
publisher_embeddings_path = '../src/publisher_embedding/data/embeddings_to_pick/sites_embeddings_red_70.pkl'
publisher_embeddings = pickle.load(open(publisher_embeddings_path, 'rb'))

### Filtro publisher da simulare

In [73]:
# Read 1 run file of an experiment
exp_path = '../results/FP_Truthful_Oracle_sigmoids_linucb_rescaledtextemb_ctr_0_9_10_12_24_nuovipub/'
run_path = exp_path + 'agent_stats_run_0_ctr_0.9_alpha_1.csv'
run_df = pd.read_csv(run_path)
# Take only the publishers of last iteration
last_iter = 0
pub_list = run_df[run_df['Iteration']==last_iter]['publisher'].tolist()
print(f"Number of publishers: {len(pub_list)}")

Number of publishers: 287


In [95]:
# Convert embeddings to numpy arrays
publisher_embeddings_array = np.array(list(publisher_embeddings.values()))
adv_embeddings_array = np.array(list(selected_adv_embeddings.values()))
# Compute scalar products between publishers and advertisers
scalar_products = np.dot(publisher_embeddings_array, adv_embeddings_array.T)
# Compute mean and std over all scalar products
mean_scores = np.mean(scalar_products)
std_scores = np.std(scalar_products)
# Compute sigmoids, i.e., the probability of clicking for each publisher-advertiser pair
sigmoids = 1 / (1 + np.exp(-(scalar_products - mean_scores) / (0.5 * std_scores)))
# The advertiser bid for each publisher is the sigmoid value for that publisher-advertiser pair
winners = np.argmax(sigmoids, axis=1)
# Filter only our sigmoids, that is the first column of the sigmoids matrix
our_sigmoids = sigmoids[:,0]
# Filter only our ctr
our_ctr = np.where((winners==0), our_sigmoids, 0)
# The number of clicks is the number of auctions per publisher (rounds_per_iter) multiplied by our ctr
clicks = our_ctr * rounds_per_iter
# Create a dataframe with the results
results = pd.DataFrame()
results['publisher'] = publisher_embeddings.keys()
results['clicks'] = clicks
results['impressions'] = rounds_per_iter
# Filter only the publishers in the pub_list (287/3398)
results = results[results['publisher'].isin(pub_list)]

In [96]:
opt_res = knapsack(results, 0.99)

Knapsack Solver: Optimal solution!
Total clicks = 5346.358647484821
Total impressions = 5400
Number of selected publishers = 27
CTR = 0.9900664162008928


In [12]:
def generate_user_contexts(
        num_iter: int, rounds_per_iter: int, dim_embedding: int, noise_strength: float, 
        publisher_embeddings: dict) -> dict:
    n, m, k = num_iter, rounds_per_iter, dim_embedding
    user_contexts = {}
    for publisher_name, pub_emb in publisher_embeddings.items():
        pub_array = np.array(pub_emb, dtype=np.float32)
        new_array = np.tile(pub_array, (n, m, 1))
        noise_array = np.random.normal(0, noise_strength, size=(n, m, k)).astype(np.float32)
        new_array += noise_array
        user_contexts[publisher_name] = new_array
    return user_contexts

def compute_sigmoids(user_contexts: dict, adv_embeddings: dict, num_iter: int, rounds_per_iter: int) -> dict:
    # Compute scalar products
    scalar_products = {}
    for publisher_name, user_context in user_contexts.items():
        scalar_products[publisher_name] = {}
        for adv_name, adv_emb in adv_embeddings.items():
            adv_array = np.array(adv_emb, dtype=np.float32)
            scalar_products[publisher_name][adv_name] = np.einsum('ijk,k->ij', user_context, adv_array)
    # Compute mean and std over all scalar products for all publishers and ads
    all_tensors = np.concatenate([score for adv_scores in scalar_products.values() for score in adv_scores.values()])
    mean_scores = np.mean(all_tensors)
    std_scores = np.std(all_tensors)
    # Compute sigmoids
    sigmoids = {}
    for publisher_name, adv_scores in scalar_products.items():
        curr_auction = np.zeros((num_iter, rounds_per_iter, len(adv_scores)))
        for i, (adv_name, score) in enumerate(adv_scores.items()):
            curr_sigmoid = 1 / (1 + np.exp(-(score - mean_scores) / (0.5 * std_scores)))
            curr_auction[:, :, i] = curr_sigmoid
        sigmoids[publisher_name] = curr_auction
    return sigmoids

def initialize_deal(
        num_iter: int, rounds_per_iter: int, dim_embedding: int, noise_strength: float, 
        publisher_embeddings: dict, adv_embeddings: dict) -> tuple[dict, dict]:
    user_contexts = generate_user_contexts(num_iter, rounds_per_iter, dim_embedding, noise_strength, publisher_embeddings)
    adv_sigmoids = compute_sigmoids(user_contexts, adv_embeddings, num_iter, rounds_per_iter)
    return user_contexts, adv_sigmoids

In [13]:
selected_publisher_embeddings = {publisher: publisher_embeddings[publisher] for publisher in pub_list}
user_contexts, sigmoids = initialize_deal(num_iter, rounds_per_iter, embedding_size, 0.01,
                                          selected_publisher_embeddings, selected_adv_embeddings)

In [14]:
# The CTR of our agent is the first value of the array of the CTRs of all agents
our_ctr = {}
for publisher_name, ctr_array in sigmoids.items():
    our_ctr[publisher_name] = ctr_array[:, :, 0]

In [15]:
# The winning bidder is the agent with the highest sigmoid value
win_bidder = {}
for publisher_name, pub_bids in sigmoids.items():
    win_bidder[publisher_name] = np.argmax(pub_bids, axis=2)

In [16]:
# The impressions are the number of times we win the auction (our bidder is in the first position)
impressions = {}
for publisher_name in sigmoids.keys():
    impressions[publisher_name] = (win_bidder[publisher_name] == 0).sum(axis=1).mean()

In [17]:
# The clicks are the CTR of the won auctions by our agent
clicks = {}
for publisher_name in sigmoids.keys():
    clicks[publisher_name] = np.where((win_bidder[publisher_name] == 0), our_ctr[publisher_name], 0).sum(axis=1).mean()

In [18]:
# Create a dataframe with the results
results = pd.DataFrame({
    'publisher': list(impressions.keys()),
    'impressions': list(impressions.values()),
    'clicks': list(clicks.values())
})

In [156]:
results.head()

,publisher,impressions,clicks
0,eltiempo.es,191.620,140.551145
1,altovicentinonline.it,147.412,129.539721
2,elchapuzasinformatico.com,200.000,189.811056
3,dating.lovepedia.net,97.708,14.825760
4,espabox.com,196.688,117.586631


In [19]:
results[results['clicks']>results['impressions']]

,publisher,impressions,clicks


In [159]:
opt_res_0_9 = knapsack(results, 0.9)

Knapsack Solver: Optimal solution!
Total clicks = 28056.266134262205
Total impressions = 31173.482
Number of selected publishers = 169
CTR = 0.9000042450908181


In [157]:
opt_res_0_97 = knapsack(results, 0.97)

Knapsack Solver: Optimal solution!
Total clicks = 9956.30509647739
Total impressions = 10264.109999999999
Number of selected publishers = 59
CTR = 0.970011534996935


In [21]:
opt_res_0_99 = knapsack(results, 0.99)

Knapsack Solver: Optimal solution!
Total clicks = 5825.2638172189
Total impressions = 5884.104
Number of selected publishers = 36
CTR = 0.9900001456838458
